# Task 6 · GRPO only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
%pip -q install -r requirements.txt
!python -m pip check
!nvidia-smi


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = ''  # REQUIRED: ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = ''  # auto-detect when the config contains one Google Drive remote


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y rclone
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    subprocess.run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
subprocess.run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), 'data/raw')
else:
    subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Experiment settings
Only ROOT_FOLDER_ID determines cloud storage. RUN_NAME groups your experiments under it.
Periodic checkpoints upload after each save; final adapter uploads after training finishes.
Local SSD is used for training and inference. A disconnected runtime can only recover checkpoints already uploaded.


In [ ]:
import json
USE_4BIT = True
RUN_FULL = False
RUN_NAME = 'experiment01'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, numbers, underscores or hyphens')
RUNS = Path('outputs/runs') / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
GRPO_STAGE = 'grpo' if RUN_FULL else 'grpo-smoke'
SFT = str(RUNS / SFT_STAGE)
GRPO = str(RUNS / GRPO_STAGE)
common = {'model.load_in_4bit': USE_4BIT}

def run(module, config, overrides, local_output, cloud_subdir):
    cmd = [sys.executable, '-m', module, '--config', config]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={json.dumps(value)}']
    env = dict(os.environ)
    env.update(MMCQA_RCLONE_CONFIG=str(config_path), MMCQA_DRIVE_ROOT_ID=root_id,
               MMCQA_SYNC_DESTINATION=cloud(cloud_subdir))
    result = subprocess.run(cmd, env=env)
    # Also preserve final adapter and logs, including after a failed training subprocess.
    if Path(local_output).exists():
        copy(local_output, cloud(cloud_subdir))
    result.check_returncode()


## GRPO only: select SFT input by run name
Paths are automatically derived from MMCQA root. This notebook never trains SFT.
A full GRPO experiment should use a full SFT checkpoint, not a smoke adapter.


In [ ]:
SFT_RUN_NAME = RUN_NAME
INPUT_SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
SFT_CHECKPOINT = 'final'  # or checkpoint-N
GRPO_OVERRIDES = {}  # e.g. {'reward.device': 'cuda', 'training.learning_rate': 0.000005}
RESUME_CHECKPOINT = ''  # GRPO checkpoint-N in this output run, for same-stage resume
local_adapter = Path('outputs/inputs/sft') / SFT_RUN_NAME / INPUT_SFT_STAGE / SFT_CHECKPOINT
copy(cloud(f'task6-runs/{SFT_RUN_NAME}/{INPUT_SFT_STAGE}/{SFT_CHECKPOINT}'), local_adapter)
if not (local_adapter / 'adapter_config.json').is_file():
    raise FileNotFoundError('SFT adapter not found. Check the selected run/stage/checkpoint.')
if not any((local_adapter / name).is_file() for name in ['adapter_model.safetensors', 'adapter_model.bin']):
    raise FileNotFoundError('SFT weights missing; complete SFT save/upload first')
grpo_args = {**common, 'model.adapter': str(local_adapter), 'model.adapter_revision': None,
             'training.output_dir': GRPO}
if not RUN_FULL:
    grpo_args.update({'data.limit': 8, 'training.max_steps': 1, 'training.save_steps': 1})
if RESUME_CHECKPOINT:
    copy(cloud(f'task6-runs/{RUN_NAME}/{GRPO_STAGE}'), GRPO)
    grpo_args['resume_from_checkpoint'] = str(Path(GRPO) / RESUME_CHECKPOINT)
grpo_args.update(GRPO_OVERRIDES)
run('mmcultureqa.training.grpo', 'configs/grpo.yaml', grpo_args, GRPO, f'task6-runs/{RUN_NAME}/{GRPO_STAGE}')


In [ ]:
if not (Path(GRPO) / 'final/adapter_config.json').is_file():
    raise FileNotFoundError('No final GRPO adapter was saved')
print('GRPO adapter on Drive:', cloud(f'task6-runs/{RUN_NAME}/{GRPO_STAGE}/final'))
